# 001 · Threads

> **Standard Library › Concurrency** · Python 3.14

The `threading` module lets you run several pieces of code "at the same time" inside one process. It is the direct counterpart of `System.Threading.Thread` — but because of CPython's **Global Interpreter Lock (GIL)**, threads behave very differently for CPU-heavy work than you are used to in .NET.

**In this notebook you will learn:**

- how to create, start and join threads (`target`/`args`, subclassing `Thread`)
- daemon threads vs .NET background threads
- identifying threads: `current_thread()`, `get_ident()`, names
- the GIL: why threads speed up I/O-bound work but not CPU-bound pure-Python code — and the free-threaded `3.14t` build
- per-thread state with `threading.local` (vs `ThreadLocal<T>` / `[ThreadStatic]`)
- running something later with `threading.Timer`

## .NET mental model

| .NET | Python `threading` |
|---|---|
| `new Thread(Work).Start()` | `threading.Thread(target=work).start()` |
| `thread.Join()` | `t.join()` |
| `thread.IsBackground = true` | `Thread(..., daemon=True)` |
| `Thread.CurrentThread` / `ManagedThreadId` | `threading.current_thread()` / `threading.get_ident()` |
| `ThreadLocal<T>` / `[ThreadStatic]` | `threading.local()` |
| `System.Threading.Timer` (one-shot) | `threading.Timer(delay, fn)` |
| `Task.Run(...)` + `await`/`.Result` | `concurrent.futures.ThreadPoolExecutor` *(see Standard Library › Concurrency › 003 · Executors with concurrent.futures)* |

Python threads **are real OS threads** (like .NET threads). The difference: in the standard CPython build only **one thread executes Python bytecode at a time** (the GIL). Threads still run truly in parallel while they are *waiting* — on sockets, files, `time.sleep`, or C code that releases the GIL.

In modern C# you rarely create a `Thread` directly; you use `Task.Run` and the thread pool. Python is the same: raw `Thread` objects are good for understanding and for long-lived background workers, but for "run these jobs and collect results" prefer an executor.

## Creating and starting a thread

```csharp
// C#
var t = new Thread(() => Greet("alice", 2));
t.Start();
t.Join();   // wait for it to finish
```

In Python you pass the callable as `target=` and its arguments as `args=` (a tuple) and/or `kwargs=` (a dict). Nothing runs until you call `.start()`.

In [ ]:
import threading
import time

def greet(name, times):
    for i in range(times):
        print(f"hello {name} #{i}")
        time.sleep(0.01)

t = threading.Thread(target=greet, args=("alice", 2))
t.start()          # begins running greet() on a new OS thread
t.join()           # block until it finishes (C#: t.Join())
print("thread finished, alive?", t.is_alive())

⚠️ **Gotcha:** `args` must be a **tuple**. `args=("alice")` is just a string in parentheses, so the thread would receive `'a', 'l', 'i', ...` as separate arguments. Write `args=("alice",)` with the trailing comma.

⚠️ **Gotcha:** `Thread(target=greet())` (with parentheses) calls `greet` **immediately on the current thread** and passes its return value (`None`) as the target. Pass the function object, not a call.

## Running several threads and collecting results

A thread's target **cannot return a value** to `join()` — there is no `Thread<T>`. The simplest pattern is to write into a shared structure. Because several threads print/finish in an unpredictable order, we collect results first and print them **sorted** afterwards.

In [ ]:
results = {}

def square(n):
    time.sleep(0.05)          # pretend to do I/O
    results[n] = n * n         # a single dict assignment is safe under the GIL

threads = [threading.Thread(target=square, args=(n,)) for n in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(sorted(results.items()))

**Tip:** if you need return values (and exceptions!), use `ThreadPoolExecutor.submit()`, which gives you a `Future` — Python's `Task<T>`. That is covered in *Standard Library › Concurrency › 003 · Executors with concurrent.futures*.

## Naming and identifying threads

```csharp
// C#
var t = new Thread(Work) { Name = "worker-1" };
Console.WriteLine(Thread.CurrentThread.ManagedThreadId);
```

Every thread has a `name` (auto-generated as `Thread-N (target)` if you don't give one) and an `ident` (an integer assigned when it starts). `threading.current_thread()` returns the `Thread` object you are running on.

In [ ]:
seen = []

def who_am_i():
    me = threading.current_thread()
    seen.append((me.name, me.ident == threading.get_ident()))

workers = [threading.Thread(target=who_am_i, name=f"worker-{i}") for i in range(3)]
for w in workers: w.start()
for w in workers: w.join()

print(sorted(seen))
print("this cell runs on:", threading.current_thread().name)
print("is main thread?", threading.current_thread() is threading.main_thread())

## Daemon threads (background threads)

```csharp
// C#
var t = new Thread(Poll) { IsBackground = true };   // won't keep the process alive
```

A **daemon** thread is Python's background thread: when only daemon threads are left, the interpreter exits and they are killed abruptly (no `finally` blocks, no flushing). Non-daemon threads (the default) keep the process alive until they finish.

In [ ]:
def heartbeat(stop_after):
    for _ in range(stop_after):
        time.sleep(0.02)

d = threading.Thread(target=heartbeat, args=(3,), daemon=True)
print("daemon?", d.daemon)
d.start()
print("alive right after start?", d.is_alive())
d.join()
print("alive after join?", d.is_alive())

⚠️ **Gotcha:** use daemon threads only for work that is safe to lose (polling, heartbeats, a dev server). Never for writing files or committing transactions. For clean shutdown, signal the thread with an `Event` and `join()` it *(see Standard Library › Concurrency › 002 · Locks, Events and Queues)*.

In a Jupyter notebook the kernel process lives on between cells, so daemon threads keep running until you restart the kernel.

## Subclassing `Thread`

Like deriving from a worker base class in C#, you can subclass `Thread` and override `run()`. This is handy when the thread carries its own state — including a place to store a result.

In [ ]:
class Downloader(threading.Thread):
    def __init__(self, url):
        super().__init__(name=f"dl-{url}")   # always call the base __init__
        self.url = url
        self.size = None

    def run(self):                           # called on the new thread by start()
        time.sleep(0.02)                      # simulated download
        self.size = len(self.url) * 100

jobs = [Downloader(u) for u in ["a.example", "bb.example", "ccc.example"]]
for j in jobs: j.start()
for j in jobs: j.join()
print([(j.url, j.size) for j in jobs])

⚠️ **Gotcha:** call `start()`, not `run()`. Calling `run()` directly just executes the method on the **current** thread — no concurrency at all. Also, a `Thread` object can be started only once (`RuntimeError` on a second `start()`).

In [ ]:
t = threading.Thread(target=lambda: None)
t.start(); t.join()
try:
    t.start()
except RuntimeError as e:
    print(f"{type(e).__name__}: {e}")

## Exceptions inside threads

In .NET an unhandled exception on a raw `Thread` crashes the process. In Python, an exception in a thread **does not propagate** to `join()` and does not stop the program: it is passed to `threading.excepthook`, which by default prints a traceback to stderr — easy to miss.

Here we install a temporary hook to capture the failure instead of printing it:

In [ ]:
captured = []
original_hook = threading.excepthook
threading.excepthook = lambda args: captured.append(f"{args.thread.name}: {args.exc_type.__name__}: {args.exc_value}")

def fails():
    raise ValueError("bad sensor reading")

t = threading.Thread(target=fails, name="sensor")
t.start()
t.join()                       # join() returns normally — no exception here!
threading.excepthook = original_hook
print(captured)

**Tip:** this is another reason to prefer executors: `future.result()` **re-raises** the worker's exception in the caller, just like awaiting a faulted `Task`.

## The GIL: I/O-bound vs CPU-bound

The GIL is a lock inside CPython that a thread must hold to execute Python bytecode. Consequences:

- **I/O-bound** work (network, disk, `sleep`, waiting on a database): a thread releases the GIL while it waits, so other threads run. Threads give a real speed-up.
- **CPU-bound pure-Python** work (loops, arithmetic on Python objects): only one thread makes progress at a time. Threads give **no** speed-up (often a small slowdown from switching).

Let's measure. First, five "I/O" jobs that each sleep 0.2 s:

In [ ]:
def io_job():
    time.sleep(0.2)

start = time.perf_counter()
for _ in range(5):
    io_job()
sequential = time.perf_counter() - start

start = time.perf_counter()
threads = [threading.Thread(target=io_job) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
threaded = time.perf_counter() - start

print(f"sequential: {sequential:.2f}s   threaded: {threaded:.2f}s")   # ~1.0s vs ~0.2s

Now a CPU-bound pure-Python function, run twice sequentially and then on two threads. (Exact timings depend on your machine; the *ratio* is what matters.)

In [ ]:
import sys

def cpu_job(n=4_000_000):
    total = 0
    for i in range(n):
        total += i * i
    return total

start = time.perf_counter()
cpu_job(); cpu_job()
sequential = time.perf_counter() - start

start = time.perf_counter()
threads = [threading.Thread(target=cpu_job) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()
threaded = time.perf_counter() - start

gil = sys._is_gil_enabled() if hasattr(sys, "_is_gil_enabled") else True
print(f"GIL enabled: {gil}")
print(f"sequential: {sequential:.2f}s   2 threads: {threaded:.2f}s   speed-up: {sequential / threaded:.1f}x")

With the GIL enabled the speed-up is about **1.0x** — two threads take as long as doing the work one after the other. In C# the same code with two threads (or `Parallel.For`) would run roughly twice as fast on a multi-core machine, because the CLR has no GIL.

For CPU-bound work in standard CPython, use **processes** *(see Standard Library › Concurrency › 004 · Processes with multiprocessing)*, or push the hot loop into C code that releases the GIL (NumPy, `hashlib` on large buffers, `zlib`, ...).

## Free-threaded Python (`3.14t`)

PEP 703 added an optional **free-threaded** build of CPython with the GIL removed: experimental in 3.13, and in 3.14 an officially supported (but still optional, non-default) build. It is a separate interpreter, usually named `python3.14t`; with uv you can install it with `uv python install 3.14t`.

- `sys._is_gil_enabled()` (3.13+) tells you whether the GIL is active right now.
- `sysconfig.get_config_var("Py_GIL_DISABLED")` tells you whether the interpreter was **built** free-threaded.

On a free-threaded build the CPU-bound demo above shows a real ~2x speed-up — threads finally behave like .NET threads. The trade-offs: single-threaded code is somewhat slower, and C extensions must explicitly declare support (otherwise the GIL is re-enabled at import time).

In [ ]:
import sysconfig

print("built free-threaded:", bool(sysconfig.get_config_var("Py_GIL_DISABLED")))
print("GIL enabled now:   ", sys._is_gil_enabled())

⚠️ **Gotcha:** the GIL never made *your* code thread-safe. It protects the interpreter's internals, not your invariants: a read-modify-write such as `counter += 1` can still interleave between threads. Use locks for shared mutable state *(see Standard Library › Concurrency › 002 · Locks, Events and Queues)* — and on a free-threaded build this matters even more.

## Thread-local data

```csharp
// C#
static ThreadLocal<string> RequestId = new(() => "none");
// or
[ThreadStatic] static string? requestId;
```

`threading.local()` creates an object whose **attributes are separate for every thread**. Each thread sees only the values it set itself.

In [ ]:
ctx = threading.local()
ctx.request_id = "main"          # set on the main thread
observed = []

def handle(request_id):
    ctx.request_id = request_id   # each thread gets its own attribute
    time.sleep(0.01)              # other threads run here and set their own value
    observed.append((request_id, ctx.request_id))

ts = [threading.Thread(target=handle, args=(f"req-{i}",)) for i in range(3)]
for t in ts: t.start()
for t in ts: t.join()

print(sorted(observed))                # every thread saw its own value
print("main still sees:", ctx.request_id)

Attributes set on one thread simply don't exist on another (like a `[ThreadStatic]` field, which starts at its default value on each new thread):

In [ ]:
def peek():
    print("in new thread, has request_id?", hasattr(ctx, "request_id"))

t = threading.Thread(target=peek)
t.start(); t.join()

**Tip:** for async code, use `contextvars.ContextVar` instead — it is the equivalent of `AsyncLocal<T>` and follows the logical flow across `await`s, whereas `threading.local` is tied to the OS thread.

## Running something later with `Timer`

`threading.Timer(interval, function, args=...)` is a `Thread` subclass that waits `interval` seconds and then calls the function once — similar to a one-shot `System.Threading.Timer`. `cancel()` stops it if it hasn't fired yet.

In [ ]:
fired = []

timer = threading.Timer(0.1, fired.append, args=("ping",))
timer.start()
timer.join()                     # it's a Thread, so we can join it
print("fired:", fired)

late = threading.Timer(0.3, fired.append, args=("never",))
late.start()
late.cancel()                    # cancelled before the 0.3s elapsed
late.join()
print("fired:", fired)

⚠️ **Gotcha:** `Timer` is one-shot and each instance is a whole OS thread. For a repeating timer, loop in a thread with `event.wait(interval)`; for many timers in async code, use `asyncio` *(see Standard Library › Concurrency › 005 · asyncio in Depth)*.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Start a thread | `new Thread(Work).Start()` | `Thread(target=work, args=(...)).start()` |
| Wait for it | `t.Join()` | `t.join()` (optionally `timeout=`) |
| Background thread | `IsBackground = true` | `daemon=True` |
| Current thread | `Thread.CurrentThread` | `threading.current_thread()` |
| Thread id | `ManagedThreadId` | `threading.get_ident()` / `t.ident` |
| Custom thread class | derive / wrap | subclass `Thread`, override `run()` |
| Unhandled exception | crashes process | goes to `threading.excepthook`; `join()` doesn't raise |
| Thread-local storage | `ThreadLocal<T>`, `[ThreadStatic]` | `threading.local()` |
| One-shot timer | `System.Threading.Timer` | `threading.Timer(sec, fn)` |
| CPU parallelism | threads / `Parallel.For` | processes (or a `3.14t` free-threaded build) |
| Check for GIL | n/a | `sys._is_gil_enabled()` |

## Exercises

1. Start 4 threads that each compute `sum(range(n))` for `n` in `[10, 100, 1000, 10000]` and store the result in a dict keyed by `n`. Print the dict sorted by key.
2. Write a `Thread` subclass `Countdown(n)` whose `run()` appends `n, n-1, ..., 1` to `self.values`. Start two of them (`3` and `2`), join both and print their values.
3. Use `threading.local()` so that each of 3 threads has its own `counter` attribute, increments it 5 times, and records `(thread name, counter)` in a list. Print the sorted list — every counter should be 5.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import threading

# 1 — results written into a shared dict, printed sorted
sums = {}
def compute(n):
    sums[n] = sum(range(n))

ts = [threading.Thread(target=compute, args=(n,)) for n in [10, 100, 1000, 10000]]
for t in ts: t.start()
for t in ts: t.join()
print(sorted(sums.items()))

# 2 — subclass with state, override run()
class Countdown(threading.Thread):
    def __init__(self, n):
        super().__init__()
        self.n = n
        self.values = []

    def run(self):
        for i in range(self.n, 0, -1):
            self.values.append(i)

a, b = Countdown(3), Countdown(2)
a.start(); b.start()
a.join(); b.join()
print(a.values, b.values)

# 3 — thread-local counter
local = threading.local()
records = []
def work():
    local.counter = 0
    for _ in range(5):
        local.counter += 1
    records.append((threading.current_thread().name, local.counter))

ts = [threading.Thread(target=work, name=f"t{i}") for i in range(3)]
for t in ts: t.start()
for t in ts: t.join()
print(sorted(records))
```

</details>